# 02 — Anotación y Visualización

## ¿Qué vamos a construir hoy?

Al final de este notebook sabrás visualizar detecciones de 5 formas diferentes
y combinarlas en capas personalizadas.

**Aprenderás a:**
- Usar el catálogo completo de Annotators de Supervision
- Personalizar colores, grosores y tamaños de texto
- Componer múltiples annotators en capas

**Tiempo estimado:** 25 minutos

## ⏱️ Estructura de la Clase (Duración estimada: 1 hora)
- **Introducción y Conceptos Base**: 15 min
- **Desarrollo y Demostración Práctica**: 25 min
- **Análisis y Casos Extremos (Pausa y Observa)**: 20 min

## Los Annotators como capas

Imagina que tienes una foto y quieres editarla en Canva o Photoshop:
añades una capa con texto, otra con un marco, otra con sombra.
Los annotators funcionan igual: cada uno añade un elemento visual diferente,
y se aplican uno sobre otro en el orden que tú eliges.

```
image.copy()
    → BoxAnnotator        → imagen con cajas
    → LabelAnnotator      → imagen con cajas + etiquetas
    → HaloAnnotator       → imagen con cajas + etiquetas + halos
```

**El orden importa:** el último annotator queda "encima" visualmente.

In [ ]:
# Instalación (descomenta si es necesario)
!pip install supervision ultralytics
import supervision as sv
from ultralytics import YOLO
import cv2
import numpy as np
import matplotlib.pyplot as plt
import urllib.request
from pathlib import Path

Path("assets").mkdir(exist_ok=True)
urllib.request.urlretrieve("https://ultralytics.com/images/bus.jpg", "assets/bus.jpg")

model = YOLO("yolov8n.pt")
image = cv2.imread("assets/bus.jpg")
results = model(image)[0]
detections = sv.Detections.from_ultralytics(results)

labels = [
    f"{results.names[c]} {conf:.0%}"
    for c, conf in zip(detections.class_id, detections.confidence)
]
print(f"Pipeline listo: {len(detections)} detecciones")

## Pausa y observa: ¿Qué hay en `detections` antes de visualizarlas?

Antes de aplicar cualquier annotator, conviene inspeccionar qué información tenemos.
Eso ayuda a decidir qué annotators tienen sentido para este conjunto de detecciones.

In [ ]:
# Inspección rápida del estado intermedio del pipeline
print(f"Objetos detectados: {len(detections)}")
print(f"Clases únicas:      {sorted(set(detections.class_id))}")
print(f"Confianza mínima:   {detections.confidence.min():.1%}")
print(f"Confianza máxima:   {detections.confidence.max():.1%}")

# Las primeras etiquetas que se usarán en los annotators de texto
# Imprimirlas ayuda a anticipar cómo se verán en la imagen
print(f"\nPrimeras 3 etiquetas: {labels[:3]}")

## El catálogo de Annotators

Supervision incluye muchos tipos de annotators. Aquí están los más usados:

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Cada annotator se instancia una sola vez y se reutiliza
# Crear el annotator dentro de un bucle reinicializa su estado en cada iteración
configs = [
    ("BoxAnnotator",       sv.BoxAnnotator()),
    ("RoundBoxAnnotator",  sv.RoundBoxAnnotator()),
    ("HaloAnnotator",      sv.HaloAnnotator()),
    ("BlurAnnotator",      sv.BlurAnnotator()),
    ("BoxCornerAnnotator", sv.BoxCornerAnnotator()),
    ("Box + Label (combo)", None),
]

for ax, (name, annotator) in zip(axes.flat, configs):
    if name == "Box + Label (combo)":
        # Demostración de composición: el resultado de uno es la entrada del siguiente
        scene = sv.BoxAnnotator().annotate(scene=image.copy(), detections=detections)
        scene = sv.LabelAnnotator().annotate(scene=scene, detections=detections, labels=labels)
    else:
        scene = annotator.annotate(scene=image.copy(), detections=detections)
    ax.imshow(cv2.cvtColor(scene, cv2.COLOR_BGR2RGB))
    ax.set_title(name, fontsize=12)
    ax.axis("off")

plt.suptitle("Catálogo de Annotators en Supervision", fontsize=14)
plt.tight_layout()
plt.show()

## Personalización: color y grosor

In [ ]:
# sv.Color tiene colores predefinidos accesibles como atributos
anotador_rojo   = sv.BoxAnnotator(color=sv.Color.RED,   thickness=3)
anotador_verde  = sv.BoxAnnotator(color=sv.Color.GREEN, thickness=3)
# sv.ColorPalette.DEFAULT asigna automáticamente un color diferente a cada clase
# — útil cuando tienes muchas categorías y no quieres asignarlas manualmente
anotador_paleta = sv.BoxAnnotator(color=sv.ColorPalette.DEFAULT, thickness=3)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (titulo, ann) in zip(axes, [
    ("Color.RED",            anotador_rojo),
    ("Color.GREEN",          anotador_verde),
    ("ColorPalette.DEFAULT", anotador_paleta),
]):
    scene = ann.annotate(scene=image.copy(), detections=detections)
    ax.imshow(cv2.cvtColor(scene, cv2.COLOR_BGR2RGB))
    ax.set_title(titulo)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 🔧 Exploración interactiva

### Experimento 1: Grosor de las cajas

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, thickness in zip(axes, [1, 4, 10]):
    ann = sv.BoxAnnotator(thickness=thickness)
    scene = ann.annotate(scene=image.copy(), detections=detections)
    ax.imshow(cv2.cvtColor(scene, cv2.COLOR_BGR2RGB))
    ax.set_title(f"thickness={thickness}")
    ax.axis("off")
plt.tight_layout()
plt.show()
# 💭 Reflexión: ¿Qué grosor resulta más legible?
# Para imágenes de alta resolución generalmente se necesita más grosor.
# Para pantallas pequeñas, menos grosor evita que las cajas tapen el objeto.

### Experimento 2: Tamaño del texto de las etiquetas

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, scale in zip(axes, [0.3, 0.6, 1.0]):
    box_ann   = sv.BoxAnnotator()
    label_ann = sv.LabelAnnotator(text_scale=scale)
    scene = box_ann.annotate(scene=image.copy(), detections=detections)
    scene = label_ann.annotate(scene=scene, detections=detections, labels=labels)
    ax.imshow(cv2.cvtColor(scene, cv2.COLOR_BGR2RGB))
    ax.set_title(f"text_scale={scale}")
    ax.axis("off")
plt.tight_layout()
plt.show()
# 💭 Reflexión: ¿text_scale=0.3 es legible en esta imagen?
# El tamaño adecuado depende de la resolución y del tamaño de los objetos detectados.

### Experimento 3: El orden de las capas importa

In [ ]:
box_ann   = sv.BoxAnnotator(thickness=3)
label_ann = sv.LabelAnnotator()

# Orden A: Box primero, Label encima
orden_a = box_ann.annotate(scene=image.copy(), detections=detections)
orden_a = label_ann.annotate(scene=orden_a, detections=detections, labels=labels)

# Orden B: Label primero, Box encima
# La caja del Box tapa parte del texto de Label
orden_b = label_ann.annotate(scene=image.copy(), detections=detections, labels=labels)
orden_b = box_ann.annotate(scene=orden_b, detections=detections)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.imshow(cv2.cvtColor(orden_a, cv2.COLOR_BGR2RGB))
ax1.set_title("Box → Label (recomendado)")
ax1.axis("off")
ax2.imshow(cv2.cvtColor(orden_b, cv2.COLOR_BGR2RGB))
ax2.set_title("Label → Box")
ax2.axis("off")
plt.tight_layout()
plt.show()
# 💭 Reflexión: ¿Por qué "Box → Label" es más legible?
# Label dibuja el texto encima de lo que ya existe en la imagen.
# Si Box va después, su línea puede tapar parte del texto.

## 🚀 Reto de extensión

**Tarea:** Crea tu combinación de annotators favorita usando al menos 3 annotators diferentes.

Explora la lista completa en: https://supervision.roboflow.com/latest/detection/annotators/

**Pista:** Prueba `sv.DotAnnotator`, `sv.TriangleAnnotator`, o `sv.EllipseAnnotator`.
Cada uno acepta los mismos parámetros base: `color=` y posiblemente `thickness=` o `radius=`.

In [ ]:
# Crea tu combinación aquí
scene = image.copy()
# scene = sv.??Annotator().annotate(scene=scene, detections=detections)
# scene = sv.??Annotator().annotate(scene=scene, detections=detections)
# scene = sv.??Annotator().annotate(scene=scene, detections=detections, labels=labels)

plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(scene, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("Mi combinación personalizada")
plt.show()